# Colab BrainIAC configuration acceptance
Run after authorized environment setup. Reads project configuration and creates one unique preflight report in Drive. No MRI/weight download, inference, or training.

In [1]:
from pathlib import Path
import json, subprocess, time
root = Path('/content/MRI-classification')
python = Path('/content/venvs/brainiac/bin/python')
assert python.is_file()
result = subprocess.run([str(python), str(root/'scripts/preflight.py'), '--encoder', 'brainiac', '--paths', str(root/'configs/paths.local.yaml'), '--config', str(root/'configs/pilot.local.yaml')], cwd=root, capture_output=True, text=True)
assert result.returncode in (0, 2), result.stderr
report = json.loads(result.stdout)
assert report['encoder'] == 'brainiac'
assert not any('snapshot verification failed' in x.lower() for x in report['missing']), report
stamp = time.strftime('%Y%m%d_%H%M%S', time.gmtime()) + '_' + str(time.time_ns())
output = Path('/content/drive/MyDrive/FYP_MRI/artifacts/reports') / ('brainiac_preflight_' + stamp + '.json')
output.parent.mkdir(parents=True, exist_ok=True)
with output.open('x') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(json.dumps({'status':report['status'], 'missing':report['missing'], 'report':str(output)}, ensure_ascii=False, indent=2))
print('No inference/training performed. Brain-JEPA environment and input contract remain unverified.')


{
  "status": "blocked",
  "missing": [
    "Set paths.brainiac_checkpoint",
    "Confirm brainiac.checkpoint_sha256",
    "Confirm brainiac.template_sha256",
    "Confirm brainiac.input_recipe_confirmed"
  ],
  "report": "/content/drive/MyDrive/FYP_MRI/artifacts/reports/brainiac_preflight_20261108_105155_1794135115660813875.json"
}
No inference/training performed. Brain-JEPA environment and input contract remain unverified.
